# جلسهٔ ۰۲ — جست‌وجوی ناآگاهانه و آگاهانه

**سطح متوسط | قابل اجرا روی CPU | آفلاین | بدون نیاز به کلید API**

الگوریتم‌های جست‌وجو مسئله‌هایی را حل می‌کنند که وضعیت‌ها و اقدام‌های ممکن آن‌ها به‌صورت دقیق تعریف شده‌اند. کیفیت مسیر پیدا‌شده و تعداد وضعیت‌های بررسی‌شده دو معیار مهم برای مقایسهٔ روش‌های جست‌وجو هستند.

### اهداف یادگیری
- صورت‌بندی مسئله به‌شکل فضای حالت (State Space) را توضیح دهید و برای آن مثالی از هوش مصنوعی بزنید.
- جست‌وجوی سطح‌اول (BFS) و جست‌وجوی عمق‌اول (DFS) را توضیح دهید و تفاوت آن‌ها را در یک مثال نشان دهید.
- جست‌وجوی با هزینهٔ یکنواخت (Uniform-Cost Search) را توضیح دهید و کاربرد آن را بشناسید.
- الگوریتم A* و تابع ابتکاری (Heuristic) را توضیح دهید و در یک مسئله به کار ببرید.

پیش از شروع، **README.md** اصلی دوره را بخوانید. قبل از اجرای هر سلول، خروجی آن را پیش‌بینی کنید.

## مثال حل‌شده

یک شبکهٔ ۵×۵ در نظر بگیرید که بعضی خانه‌های آن مسدود هستند. خانهٔ شروع و هدف را مشخص کنید. سپس برای وضعیت‌های مرزی، مقدارهای زیر را در نظر بگیرید:

- `g(n)`: هزینهٔ مسیری که از شروع تا وضعیت `n` طی شده است
- `h(n)`: برآورد هزینهٔ باقی‌مانده از `n` تا هدف
- `f(n) = g(n) + h(n)`: امتیازی که A* برای اولویت‌بندی وضعیت‌ها استفاده می‌کند

توضیح دهید چرا خانه‌ای که به‌ظاهر نزدیک‌ترین فاصله را تا هدف دارد، لزوماً ارزان‌ترین مسیر را ایجاد نمی‌کند.

**پرسش کلاسی:** الگوریتم BFS تحت چه شرایطی کوتاه‌ترین مسیر را پیدا می‌کند؟

## تعریف شبکه و پیاده‌سازی BFS

همسایهٔ معتبر باید در محدودهٔ شبکه باشد و در خانهٔ مسدود قرار نگیرد. در این مثال `.` نشان‌دهندهٔ خانهٔ آزاد و `#` نشان‌دهندهٔ مانع است.

الگوریتم BFS با استفاده از صف (`deque`) وضعیت‌ها را به‌ترتیب عمق بررسی می‌کند. علاوه بر مسیر یافت‌شده، تعداد وضعیت‌های گسترش‌یافته را هم ثبت می‌کنیم.

In [ ]:
from collections import deque
GRID=[".....",".###.","...#.",".#...","....."]
START,GOAL=(0,0),(4,4)
def neighbors(p):
    r,c=p
    for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)]:
        nr,nc=r+dr,c+dc
        if 0<=nr<5 and 0<=nc<5 and GRID[nr][nc]==".":
            yield (nr,nc)
def path(prev,p):
    out=[p]
    while p in prev: p=prev[p];out.append(p)
    return out[::-1]
def bfs():
    q=deque([START]); seen={START};prev={};count=0
    while q:
        cur=q.popleft();count+=1
        if cur==GOAL:return path(prev,cur),count
        for nxt in neighbors(cur):
            if nxt not in seen: seen.add(nxt);prev[nxt]=cur;q.append(nxt)
    return None,count
print("خروجی BFS:",bfs())

## الگوریتم A* و مقایسهٔ کنترل‌شده

در A* برای هر وضعیت، بهترین هزینهٔ مسیرِ شناخته‌شده (`g`) را نگه می‌داریم. با صف اولویت (`heapq`) وضعیتی را بررسی می‌کنیم که کوچک‌ترین `f = g + h` را دارد.

در این مثال، `h` فاصلهٔ منهتن (Manhattan Distance) تا هدف است؛ چون حرکت‌ها چهارجهته و هزینهٔ هر حرکت یکسان است، این تابع هزینهٔ باقی‌مانده را بیش‌برآورد نمی‌کند.

برای مقایسهٔ منصفانه، BFS و A* را روی یک شبکه و با نقطهٔ شروع و هدف یکسان اجرا کنید. به طول مسیر و تعداد وضعیت‌های گسترش‌یافته توجه کنید.

In [ ]:
from heapq import heappush,heappop
def astar():
    h=lambda q:abs(q[0]-GOAL[0])+abs(q[1]-GOAL[1])
    heap=[(h(START),0,START)];best={START:0};prev={};count=0
    while heap:
        _,g,cur=heappop(heap)
        if g!=best[cur]:continue
        count+=1
        if cur==GOAL:return path(prev,cur),count
        for nxt in neighbors(cur):
            new=g+1
            if new<best.get(nxt,float("inf")):
                best[nxt]=new;prev[nxt]=cur;heappush(heap,(new+h(nxt),new,nxt))
    return None,count
b,be=bfs();a,ae=astar()
print("طول مسیر:",len(a)-1,"وضعیت‌های گسترش‌یافته در BFS:",be,"وضعیت‌های گسترش‌یافته در A*:",ae)
assert b is not None and len(a)==len(b)

## تمرین

جای موانع (`#`) را تغییر دهید و تعداد وضعیت‌های گسترش‌یافته به‌وسیلهٔ BFS و A* را مقایسه کنید. آیا هر دو مسیر هم‌طول پیدا می‌کنند؟ چرا؟

**تکلیف:** جست‌وجوی با هزینهٔ یکنواخت (Uniform-Cost Search) را برای شبکه‌ای پیاده‌سازی کنید که حرکت به بعضی خانه‌ها گران‌تر است.

**برداشت نادرست رایج:** BFS روی گرافی با هزینهٔ یال‌های نابرابر، لزوماً مسیر کم‌هزینه را پیدا نمی‌کند.

نوت‌بوک را در یک کرنل تازه و از ابتدا اجرا کنید و دست‌کم یک محدودیت روش‌های استفاده‌شده را توضیح دهید.